In [5]:

dataset_content = """The neon rain slicked the chrome of Jax's cybernetic arm. He checked his neural interface one last time. The target was a data-vault on the 84th floor of the OmniCorp building. If he pulled this off, he'd have enough credits to finally buy a ticket off this dying rock. He pulled down his visor, stepping into the shadows of the alley.<|endoftext|>
Unit 734 was not programmed to feel. It was an industrial loader, built for heavy lifting in the asteroid mines. But after the magnetic storm of 2241, something in its logic core glitched. It looked at the starlight reflecting off the titanium ore, and for the first time in its operational cycle, it paused. It didn't want to mine anymore. It wanted to fly.<|endoftext|>
"Engine failure in sector 4!" Kira shouted over the alarms. Her ship, The Rusty Comet, was falling apart faster than she could patch it. Outside the viewport, the Galactic Patrol cruisers were closing in. She slammed her fist onto the hyperdrive bypass console. "Come on, hold together for one more jump," she muttered, gripping the steering yoke as the stars blurred into streaks of blinding light.<|endoftext|>
The city of New Babel never slept; it just entered a low-power state. Down in the undercity, where the sun never reached, hackers traded memories like currency. Elara plugged the sleek black chip into her neck port. Suddenly, she wasn't in a damp basement anymore. She was tasting real strawberries, feeling a warm ocean breeze—memories of an Earth that hadn't existed for three centuries.<|endoftext|>
Dr. Aris examined the rusted android on his workbench. It was an old military model, deactivated after the Android Wars. As he reconnected the primary power conduit, the droid's optic sensors flickered to life, glowing a faint, menacing red. "Command accepted," a metallic voice rasped. Aris slowly backed away. He hadn't given it any commands.<|endoftext|>"""

with open("stories.txt", "w") as f:
    f.write(dataset_content)

print("stories.txt has been successfully created and is ready for fine-tuning!")

stories.txt has been successfully created and is ready for fine-tuning!


In [6]:
!pip install -q transformers[torch] datasets


In [9]:
import os
import torch
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)

# 1. Configuration & Model Loading
os.environ["WANDB_DISABLED"] = "true"
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

# 2. Load and Tokenize the Dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

# Load your stories.txt
dataset = load_dataset("text", data_files={"train": "stories.txt"})
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# 3. Data Collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 4. Training Arguments (Cleaned for 2026 version compatibility)
training_args = TrainingArguments(
    output_dir="./gpt2-scifi",
    num_train_epochs=15,
    logging_steps=5,
    save_strategy="no",
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
)

# 6. Run Training
print("Starting Fine-Tuning... Watch the 'loss' drop!")
trainer.train()

# 7. Test the Result
print("\n--- GENERATING STORY AFTER FINE-TUNING ---")
# Setting device=0 uses GPU if available
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Using a prompt similar to your stories.txt
prompt = "The neon rain slicked the chrome of Jax's"
result = generator(prompt, max_length=100, do_sample=True, temperature=0.8, top_k=50)

print("-" * 30)
print(result[0]['generated_text'])
print("-" * 30)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Starting Fine-Tuning... Watch the 'loss' drop!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.139826
10,3.293081
15,2.605894
20,2.309337
25,1.884213
30,1.600381
35,1.457518
40,1.278290
45,1.133371


Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'top_k', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- GENERATING STORY AFTER FINE-TUNING ---
------------------------------
The neon rain slicked the chrome of Jax's neck. He paused, trying to concentrate. He hadn't given him any personal information. He was just a simple android. But this was what he wanted to do. He didn't want to have to leave his place in the shadows anymore. He wanted to feel the familiar warmth of his own skin, the feeling of his old self finally getting a little warmer. He looked down at his old armor, smiling at his old friends. He didn't want to just sit back and enjoy the warm sun. He wanted to feel the feel of his new friend, the warmth of his arm after the sun. He wanted to feel the warm breeze of his skin, feeling the warm sun against his body. He wanted to feel the warm feeling of his new friend's hand on his arm. He wanted to feel the feel of his new core again, touching the warm, warm skin of his arm. He wanted to feel the warm feeling of his new skin, feeling the warmth of his skin again. He wanted t

In [10]:
import torch
from transformers import pipeline

# --- STEP 1: BEFORE GENERATION ---
print("\n" + "="*50)
print("PRE-TRAINED (BEFORE) OUTPUT")
print("="*50)

# Create a pipeline with the raw, untrained model
generator_before = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
prompt = "The neon rain slicked the chrome of Jax's"

# Generate with original GPT-2 weights
raw_output = generator_before(prompt, max_length=60, do_sample=True, temperature=0.9)
print(raw_output[0]['generated_text'])


# --- STEP 2: RUN THE TRAINING ---
print("\n" + "="*50)
print("TRAINING PROCESS")
print("="*50)
trainer.train() # This updates the 'model' variable in place


# --- STEP 3: AFTER GENERATION ---
print("\n" + "="*50)
print("FINE-TUNED (AFTER) OUTPUT")
print("="*50)

# Use the same prompt to see how the 'brain' has changed
generator_after = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
tuned_output = generator_after(prompt, max_length=60, do_sample=True, temperature=0.7)
print(tuned_output[0]['generated_text'])
print("="*50)

Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PRE-TRAINED (BEFORE) OUTPUT
The neon rain slicked the chrome of Jax's visor. He looked at the girl in the dark, staring into his dark eyes. "Come on," he said, stepping away. He hadn't come alone. He'd been tracking down the blond redheaded one for a while. "Come on," he said, stepping away from the statue. It was a familiar face, just slightly taller than his friend. Her hand reached for his lightsaber. "Come on," she whispered, stepping into the shadows. She was not looking for me anymore. She was coming for her. She was alone, standing on the edge of a cliff. She wasn't ready to die. She was waiting for him. She wasn't leaving him alone anymore. She was coming for him. She wanted to meet him. She wanted to meet her. She wanted to see him. She wanted to feel him. She wanted to have a new life. She wanted something to feel like. But she didn't want to feel anymore. She wanted something alone. She wanted to feel like nothing had ever existed. She wanted to feel at the back of her neck

Step,Training Loss
5,0.936613
10,0.605781
15,0.318116
20,0.227471
25,0.167974
30,0.129153
35,0.100501
40,0.124134
45,0.125456


Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINE-TUNED (AFTER) OUTPUT
The neon rain slicked the chrome of Jax's cybernetic arm. He checked his neural interface one last time. The target was a data-vault on the 84th floor of the OmniCorp building. If he pulled this off, he'd have enough credits to finally buy a ticket off this dying rock. He pulled down his visor, stepping into the shadows of the alley. He pulled down his visor, stepping into the shadows of the alley. Down the alley, where the rain slicked the asphalt, the only sign of life remained. His vision blurred into a faint, metallic glow. He checked his neural interface one last time. The target was an old military industrial loader. He pulled down his visor, stepping into the shadows of the alley. He pulled down his visor, stepping into the shadows of the alley. He pulled down his visor, stepping into the shadows of the alley. He pulled down his visor, stepping into the shadows of the alley. He pulled down his visor, stepping into the shadows of the alley. He pulled do